# Introduction to Pydantic

<br>

## DataClass `vs` BaseModel Inheritance

In [12]:
from pydantic import BaseModel

In [13]:
# Using Dataclass (BaseModel not inherited)

from dataclasses import dataclass

@dataclass
class Person1():
  name: str
  age: int
  city: str

person1 = Person1(name="Rick", age=25, city="Dallas")
print(person1)

Person1(name='Rick', age=25, city='Dallas')


In [15]:
# BaseModel Inherited

class Person2(BaseModel):
  name: str
  age: int
  city: str

person2 = Person2(name="Rick", age=25, city="Dallas")
print(person2)


name='Rick' age=25 city='Dallas'


In [16]:
type(person1)

__main__.Person1

In [17]:
type(person2)

__main__.Person2

In [18]:
person1 = Person1(name="Rick", age=25, city=3444) # Deliberately putting an error (dataclass method)
print(person1)


Person1(name='Rick', age=25, city=3444)


In [19]:
person2 = Person2(name="Rick", age=25, city=3444) # Deliberately putting an error (BaseModel method)
print(person2)

ValidationError: 1 validation error for Person2
city
  Input should be a valid string [type=string_type, input_value=3444, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [21]:
'''
Observe how BaseModel (from Pydantic) did the data validation and raising an error
Dataclass Method failed to do that
'''

'\nObserve how BaseModel (from Pydantic) did the data validation and raising an error\n'

<br>

## Model(s) with Optional Fields

In [23]:
from typing import Optional

class Employee(BaseModel):
  id: int
  name: str
  department: str
  salary: Optional[float] = None # Optional with default value (None)
  is_active: Optional[bool] = True # Optional with default value (True)

emp1 = Employee(id=1, name="Rick", department='AI')
print(emp1)

id=1 name='Rick' department='AI' salary=None is_active=True


In [25]:
emp2 = Employee(id=1, name="Rick", department='AI', salary=10000, is_active=False)
print(emp2) # Typecasting is possible (for respective data type - mentioned above as optional (here))

id=1 name='Rick' department='AI' salary=10000.0 is_active=False


In [29]:
from typing import List

class Classoom(BaseModel):
  room_number: str
  students: List[str] # List fo Strings
  capacity: int

In [30]:
classroom = Classoom(
  room_number="A123",
  students=["Rick", "Myers"],
  capacity=45
)

print(classroom)

room_number='A123' students=['Rick', 'Myers'] capacity=45


In [33]:
classroom1 = Classoom(
  room_number="A123",
  students=("Rick", "Myers"), # Typecasting will happen here (even dictionary can also get passed - will come out as list)
  capacity=45
)

print(classroom1)

room_number='A123' students=['Rick', 'Myers'] capacity=45


In [35]:
try:
  invalid_val = Classoom(room_number="A123", students=["Rick", 70], capacity=45)
except ValueError as e:
  print(e)

1 validation error for Classoom
students.1
  Input should be a valid string [type=string_type, input_value=70, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


<br>

## Model(s) with Nested Models

In [38]:
class Address(BaseModel):
  street: str
  city: str
  zip_code: str # Even if this is "int" and you give str value - it will get typecasted by Pydantic

class Customer(BaseModel):
  customer_id: int
  name: str
  address: Address # Nested Model

customer = Customer(
  customer_id=1,
  name="Emma",
  address={"street": "123 Main St", "city": "NYC", "zip_code": "12345"}
)

print(customer)

customer_id=1 name='Emma' address=Address(street='123 Main St', city='NYC', zip_code='12345')


<br>

## Pydantic Fields : Customization & Constraints

In [40]:
from pydantic import Field

class Item(BaseModel):
  name:str = Field(min_length=2, max_length=50) # Constraints on string length
  price:float = Field(gt=0, le=1000) # 0 < val <= 1000
  quantity:int = Field(ge=0) # val >= 0

item1 = Item(name="Book", price=99.34, quantity=45)
print(item1)

name='Book' price=99.34 quantity=45


In [41]:
item1 = Item(name="Book", price=-9.34, quantity=45)
print(item1)

ValidationError: 1 validation error for Item
price
  Input should be greater than 0 [type=greater_than, input_value=-9.34, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [42]:
class User(BaseModel):
  username:str = Field(..., description="Unique username for the user")
  age: int = Field(default=18, description="User Age, defaults to 18")
  email: str = Field(default_factory=lambda: "user@example", description="Default email address")

user1 = User(username="Rick")
print(user1)

user2 = User(username="Bob", age=34, email="Bob123@gmail.com")
print(user2)

username='Rick' age=18 email='user@example'
username='Bob' age=34 email='Bob123@gmail.com'


In [45]:
print(User.model_json_schema())

{'properties': {'username': {'description': 'Unique username for the user', 'title': 'Username', 'type': 'string'}, 'age': {'default': 18, 'description': 'User Age, defaults to 18', 'title': 'Age', 'type': 'integer'}, 'email': {'description': 'Default email address', 'title': 'Email', 'type': 'string'}}, 'required': ['username'], 'title': 'User', 'type': 'object'}
